# 09 — Estimate the thesis models

Existing quarterly inputs only. Run 08 first, then this notebook, then 10.
The main specification is unchanged: NB2 counts and a Fachorganisation-stratified
start–stop Cox model. H3 compares A versus C with matched Poisson QMLE models.
Tight optimizer tolerances and a Newton-correction check prevent premature convergence.
One stock-curvature check, larger spatial clusters and one age-varying Cox sensitivity
address the specific flags in the real runs. No automated model search.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import platform
import warnings
import uuid

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import scipy
from scipy import stats
from scipy.special import digamma
import statsmodels
import statsmodels.api as sm
from statsmodels.discrete.discrete_model import NegativeBinomial
from statsmodels.duration.hazard_regression import PHReg
from statsmodels.base.model import LikelihoodModel

CONFIG = {
    "cell_size": 100, "legacy_100m": False,
    "ph_sensitivity": True,
    "spatial_block_m": 5000,
    "maxiter_poisson": 300, "maxiter_nb2": 300, "maxiter_cox": 300,
    "optimizer_nb2": "lbfgs", "alpha_lower": 1e-6, "alpha_upper": 1e4,
}
START_YEAR, END_YEAR, LAG_YEAR = 2016, 2025, 2015
PROJECT_DIR = next((p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
                    if (p / "ANAL").is_dir()), None)
if PROJECT_DIR is None:
    raise RuntimeError("Open this notebook inside the project.")
CELL_SIZE = CONFIG["cell_size"]
if CELL_SIZE < 100 or CELL_SIZE % 100 or (CONFIG["legacy_100m"] and CELL_SIZE != 100):
    raise ValueError("Use a multiple of 100 m; the legacy layout is only for 100 m.")
RESOLUTION = f"{CELL_SIZE}m"
INPUT_PROFILE = "100m_legacy" if CONFIG["legacy_100m"] else RESOLUTION
DATA_DIR = PROJECT_DIR / "ANAL" / "data"
if not CONFIG["legacy_100m"]:
    DATA_DIR = DATA_DIR / RESOLUTION
FEATURE_DIR = DATA_DIR / "routing" / "features"
PANEL_PATH = DATA_DIR / f"raster_quarter_panel_{RESOLUTION}.parquet"
FIRMS_PATH = DATA_DIR / f"firms_assigned_{RESOLUTION}.geoparquet"
RASTER_PATH = DATA_DIR / f"raster_{RESOLUTION}_styria.geoparquet"
H3_BIRTHS_PATH = DATA_DIR / f"births_by_fachgruppe_{RESOLUTION}.parquet"
RESULT_DIR = DATA_DIR / "models"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:8]
BUNDLE_DIR = RESULT_DIR / "runs" / RUN_ID
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)
sample_rows, diagnostic_rows, compatibility_mappings = [], [], []
FOUNDING_FREQUENCY = "quarter"
manifest = {
    "run_id": RUN_ID, "status": "running", "pipeline_version": "09-minimal-1",
    "resolution_m": CELL_SIZE, "input_profile": INPUT_PROFILE, "config": CONFIG,
    "study_period": {"start_year": START_YEAR, "end_year": END_YEAR, "founding_lag_year": LAG_YEAR},
    "time_aggregation": {"founding": "quarter", "survival": "quarter"},
    "software": {"python": platform.python_version(), "numpy": np.__version__,
                 "pandas": pd.__version__, "scipy": scipy.__version__,
                 "statsmodels": statsmodels.__version__},
}
(RESULT_DIR / "latest_attempt.json").write_text(json.dumps({"run_id": RUN_ID, "status": "running"}))
print(f"{INPUT_PROFILE}: 2016Q1–2025Q4. Aggregate outputs: {BUNDLE_DIR}")

## 1. Prepare the two model samples

In [ ]:
def file_hash(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for block in iter(lambda: stream.read(2**20), b""):
            digest.update(block)
    return digest.hexdigest()

def notebook_code_hash(path):
    notebook = json.loads(path.read_text(encoding="utf-8"))
    source = "\n".join("".join(c["source"]) for c in notebook["cells"] if c["cell_type"]=="code")
    return hashlib.sha256(source.encode()).hexdigest()

def note(model, check, status, detail):
    diagnostic_rows.append(dict(model=model, check=check, status=status, detail=detail))

def columns(path):
    return set(pq.ParquetFile(path).schema_arrow.names)

def grid_column(path):
    names = columns(path)
    if "grid_id" in names: return "grid_id"
    if CONFIG["legacy_100m"] and "grid_id_100m" in names:
        item = {"file": str(path.relative_to(PROJECT_DIR)), "source": "grid_id_100m", "canonical": "grid_id"}
        if item not in compatibility_mappings: compatibility_mappings.append(item)
        return "grid_id_100m"
    raise ValueError(f"{path.name}: missing grid_id (legacy alias is allowed only in the legacy profile).")

def require_columns(path, required):
    available = columns(path)
    if "grid_id" in required:
        grid_column(path)
        available |= {"grid_id"}
    missing = sorted(set(required)-available)
    if missing: raise ValueError(f"{path}: missing columns {missing}")

In [ ]:
FOUNDING_TERMS = ["log_own_firms", "log_own_pop", "log_pop_access_ring_0_15", "log_firms_relative_car_ring_0_15",
                 "log_walk_pop_ring_0_10", "log_firms_relative_walk_ring_0_10", "population_growth_yoy",
                 "log_tt_motorway_exit", "walk_pt_routes_10min", "pt_ohne_haltestelle"]
SURVIVAL_TERMS = ["log_own_pop", "log_own_same", "log_own_other", "log_pop_ring_0_15", "log_same_relative_ring_0_15",
                  "log_other_relative_ring_0_15", "log_tt_motorway_exit", "walk_pt_routes_10min", "pt_ohne_haltestelle", "calendar_year"]


PREPARED_DIR = DATA_DIR / "cache" / "model_tables"
meta_path = PREPARED_DIR / "model_data_manifest.json"
if not meta_path.is_file():
    raise FileNotFoundError("Run the corresponding notebook 08 first to prepare the model tables.")
prepared = json.loads(meta_path.read_text(encoding="utf-8"))
if (prepared["status"] != "completed" or prepared["version"] != "model-tables-1"
        or prepared["resolution_m"] != CELL_SIZE or prepared["legacy_100m"] != CONFIG["legacy_100m"]):
    raise ValueError("Prepared-data profile mismatch; run notebook 08 again.")
preparation_notebook = PROJECT_DIR / "ANAL" / prepared["preparation_notebook"]
if notebook_code_hash(preparation_notebook) != prepared["code_sha256"]:
    raise ValueError("Preparation code changed; run notebook 08 again.")
for source in prepared["source_files"]:
    if file_hash(PROJECT_DIR / source["path"]) != source["sha256"]:
        raise ValueError("Source data changed; run notebook 08 again.")
for table in prepared["tables"].values():
    if file_hash(PREPARED_DIR / table["file"]) != table["sha256"]:
        raise ValueError("Prepared table changed; run notebook 08 again.")
founding = pd.read_parquet(PREPARED_DIR / prepared["tables"]["founding"]["file"])
survival_frame = pd.read_parquet(PREPARED_DIR / prepared["tables"]["survival"]["file"])
sample_rows.extend(prepared["sample_description"])
diagnostic_rows.extend(prepared["diagnostics"])
manifest["sources"] = prepared["source_files"]
manifest["code_sha256"] = notebook_code_hash(PROJECT_DIR / "ANAL" / "09_estimate_and_save_models.ipynb")
manifest["preparation_code_sha256"] = prepared["code_sha256"]
display(pd.DataFrame(sample_rows))

grid = pd.read_parquet(RASTER_PATH, columns=[grid_column(RASTER_PATH), "centroid_x", "centroid_y"])
grid = grid.rename(columns={"grid_id_100m": "grid_id"})
if grid.grid_id.isna().any() or grid.grid_id.duplicated().any():
    raise ValueError("Raster coordinates must identify cells uniquely.")
if not np.isfinite(grid[["centroid_x", "centroid_y"]]).all().all():
    raise ValueError("Missing projected raster coordinates.")
block_x = np.floor(grid.centroid_x / CONFIG["spatial_block_m"]).astype(int)
block_y = np.floor(grid.centroid_y / CONFIG["spatial_block_m"]).astype(int)
grid["spatial_block"] = block_x.astype(str) + "_" + block_y.astype(str)
block_map = grid.set_index("grid_id").spatial_block
for frame in [founding, survival_frame]:
    frame["spatial_block"] = frame.grid_id.map(block_map)
    if frame.spatial_block.isna().any():
        raise ValueError("A model cell has no spatial-block assignment.")

## 2. Gründungsmodell

Absolute counts, no offset, full year-quarter effects. The population effect is
mass minus relative firms, with the full covariance. Cell clusters allow temporal
dependence; 5 km blocks are an additional spatial-uncertainty sensitivity.

In [ ]:
def design_matrix(frame, terms, periods=False):
    X = frame[terms].astype(float)
    if periods:
        X = pd.concat([X, pd.get_dummies(frame.period, prefix="period", drop_first=True, dtype=float)], axis=1)
        X.insert(0, "const", 1.0)
    if not np.isfinite(X).all().all():
        raise ValueError("Non-finite model matrix.")
    constants = [c for c in X if c != "const" and X[c].nunique() < 2]
    if constants:
        print("Unidentified constant terms removed:", constants)
    X = X.drop(columns=constants)
    scale = X.std().replace(0, 1)
    if "const" in scale:
        scale["const"] = 1
    scaled = X / scale
    if np.linalg.matrix_rank(scaled.to_numpy().T @ scaled.to_numpy(), hermitian=True) < X.shape[1]:
        raise ValueError("Resolve exact collinearity before fitting.")
    return X, scaled, scale, constants


def check_fit_arrays(params, covariance, loglike):
    if not np.isfinite(params).all() or not np.isfinite(covariance).all() or not np.isfinite(loglike):
        raise RuntimeError("Non-finite model result.")
    if (np.diag(covariance) <= 0).any():
        raise RuntimeError("Non-positive coefficient uncertainty.")
    if np.linalg.eigvalsh(covariance).min() < -1e-8 * max(1, np.diag(covariance).max()):
        raise RuntimeError("Invalid covariance matrix.")


def numerical_check(model, params):
    information = -model.hessian(params)
    np.linalg.cholesky(information)
    bread = np.linalg.inv(information)
    if (np.diag(bread) <= 0).any():
        raise RuntimeError("Non-positive information matrix.")
    step = bread @ model.score(params)
    step_se = float(np.max(np.abs(step) / np.sqrt(np.diag(bread))))
    if not np.isfinite(step_se) or step_se > 0.05:
        raise RuntimeError(f"Premature optimizer stop: Newton correction = {step_se:.3g} standard errors.")
    return bread, step_se


def count_estimate(family, X, y, groups, scale, poisson=None, start=None):
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        if family == "poisson":
            result = sm.GLM(y, X, family=sm.families.Poisson()).fit(
                maxiter=CONFIG["maxiter_poisson"], tol=1e-10,
                cov_type="cluster", cov_kwds={"groups": groups})
            converged = result.converged
            iterations = result.fit_history.get("iteration", 0)
        else:
            if start is None:
                start = np.r_[np.asarray(poisson["params"]) * scale.to_numpy(), .1]
            result = NegativeBinomial(y, X, loglike_method="nb2").fit(
                start_params=start, method="lbfgs", maxiter=CONFIG["maxiter_nb2"],
                pgtol=1e-10, factr=100, disp=0,
                bounds=[(None, None)] * X.shape[1] +
                       [(np.log(CONFIG["alpha_lower"]), np.log(CONFIG["alpha_upper"]))],
                cov_type="cluster", cov_kwds={"groups": groups})
            converged = result.mle_retvals.get("converged", False)
            iterations = result.mle_retvals.get("iterations", 0)
        if not converged:
            raise RuntimeError(f"{family} did not converge.")
        bread, step_se = numerical_check(result.model, np.asarray(result.params))
    factor = np.r_[scale, 1.] if family == "nb2" else np.asarray(scale)
    fit = {
        "family": family, "names": [*X.columns, "alpha"] if family == "nb2" else list(X.columns),
        "params": (np.asarray(result.params) / factor).tolist(),
        "covariance": (np.asarray(result.cov_params()) / np.outer(factor, factor)).tolist(),
        "naive_covariance": (bread / np.outer(factor, factor)).tolist(),
        "log_likelihood": float(result.llf), "aic": float(result.aic),
        "converged": True, "iterations": int(iterations), "newton_step_se": step_se,
        "warnings": sorted({str(w.message) for w in caught}), "loaded_from_cache": False,
        "n_observations": len(y), "n_clusters": pd.Series(groups).nunique(),
        "covariance_method": "grid-cell cluster sandwich (statsmodels)",
    }
    check_fit_arrays(np.asarray(fit["params"]), np.asarray(fit["covariance"]), fit["log_likelihood"])
    if family == "nb2" and poisson is not None and result.llf + .01 < poisson["log_likelihood"]:
        raise RuntimeError("NB2 likelihood below Poisson; inspect optimization.")
    for message in fit["warnings"]:
        print(f"{family} warning: {message}")
    print(f"{family}: iterations={iterations}, maximum Newton correction/SE={step_se:.3g}")
    return fit


def coefficient_table(fit, effect="irr"):
    params = np.asarray(fit["params"])
    covariance = np.asarray(fit["covariance"])
    se = np.sqrt(np.diag(covariance))
    critical = stats.norm.ppf(.975)
    z = params/se
    table = pd.DataFrame({"coefficient": params, "std_error": se, "z": z, "p_value": 2*stats.norm.sf(np.abs(z)),
                          "ci_lower": params-critical*se, "ci_upper": params+critical*se}, index=fit["names"])
    table.index.name = "term"
    with np.errstate(over="ignore"):
        table[effect] = np.exp(table.coefficient)
        prefix = "irr" if effect == "irr" else "hr"
        table[prefix+"_ci_lower"] = np.exp(table.ci_lower)
        table[prefix+"_ci_upper"] = np.exp(table.ci_upper)
    if "alpha" in table.index: table.loc["alpha", [effect, prefix+"_ci_lower", prefix+"_ci_upper"]] = np.nan
    return table

def linear_effect(fit, weights):
    names = fit["names"]
    unknown = set(weights)-set(names)
    if unknown: raise ValueError(f"Contrast refers to missing/unidentified terms: {sorted(unknown)}")
    vector = np.asarray([weights.get(name, 0.) for name in names], dtype=float)
    estimate = float(vector @ np.asarray(fit["params"]))
    variance = float(vector @ np.asarray(fit["covariance"]) @ vector)
    if not np.isfinite(vector).all() or not np.isfinite(variance) or variance <= 0: raise ValueError("Contrast has non-positive variance.")
    se = float(np.sqrt(variance))
    return estimate, se

In [ ]:
POPULATION = {"log_pop_access_ring_0_15": 1, "log_firms_relative_car_ring_0_15": -1}
COUNT_TESTS = {
    "H1": (POPULATION, "positive", "tested"),
    "H2a": ({"log_firms_relative_car_ring_0_15": 1}, "positive", "tested"),
    "H4_routes": ({"walk_pt_routes_10min": 1}, "positive", "partial"),
    "H4_no_stop": ({"pt_ohne_haltestelle": 1}, "negative", "partial"),
    "H5": ({"log_tt_motorway_exit": 1}, "negative", "partial"),
}
SURVIVAL_TESTS = {
    "H2b": ({"log_same_relative_ring_0_15": 1}, "positive", "tested"),
    "H2c": ({"log_other_relative_ring_0_15": 1}, "negative", "tested"),
}


def contrast_row(identifier, estimate, variance, expected, coverage, covariance):
    if not np.isfinite(variance) or variance <= 0:
        raise ValueError(f"{identifier}: non-positive contrast variance.")
    se = float(np.sqrt(variance))
    return dict(hypothesis_id=identifier, estimate=float(estimate), std_error=se,
                ci_lower=float(estimate - 1.96 * se), ci_upper=float(estimate + 1.96 * se),
                p_value=float(2 * stats.norm.sf(abs(estimate / se))),
                expected_direction=expected,
                direction_matches=bool(estimate > 0 if expected == "positive" else estimate < 0),
                coverage=coverage, covariance=covariance)


def hypothesis_table(fit, definitions):
    rows = []
    for identifier, (weights, expected, coverage) in definitions.items():
        if set(weights) - set(fit["names"]):
            rows.append(dict(hypothesis_id=identifier, coverage="unidentified"))
            continue
        estimate, se = linear_effect(fit, weights)
        rows.append(contrast_row(identifier, estimate, se**2, expected, coverage, "cell"))
    return pd.DataFrame(rows)

In [ ]:
X_founding, X_nb_scaled, nb_scale, nb_constants = design_matrix(founding, FOUNDING_TERMS, periods=True)
y_founding = founding.births.to_numpy(dtype=float)
poisson_fit = count_estimate("poisson", X_nb_scaled, y_founding, founding.grid_id, nb_scale)
founding_fit = count_estimate("nb2", X_nb_scaled, y_founding, founding.grid_id, nb_scale, poisson_fit)
founding_coefficients = coefficient_table(founding_fit)
mu_poisson = np.exp(X_founding.to_numpy() @ np.asarray(poisson_fit["params"]))
mu_founding = np.exp(X_founding.to_numpy() @ np.asarray(founding_fit["params"])[:-1])
alpha = founding_fit["params"][-1]
pearson = (y_founding - mu_founding) / np.sqrt(mu_founding + alpha * mu_founding**2)
founding_hypotheses = hypothesis_table(founding_fit, COUNT_TESTS)
display(founding_hypotheses)
display(founding_coefficients.loc[[x for x in founding_coefficients.index if not x.startswith("period_")]])

## 3. Count-model checks

In [ ]:
comparison_rows = []
for fit, mu in [(poisson_fit, mu_poisson), (founding_fit, mu_founding)]:
    a = fit["params"][-1] if fit["family"] == "nb2" else 0
    zero = np.exp(-np.log1p(a*mu)/a) if a > 0 else np.exp(-mu)
    comparison_rows.append(dict(model=fit["family"], n_observations=len(founding),
        log_likelihood=fit["log_likelihood"], aic=fit["aic"], alpha=a,
        pearson_dispersion=float(np.sum((y_founding-mu)**2/(mu+a*mu**2))/(len(mu)-len(fit["params"]))),
        zero_observed=float(np.mean(y_founding == 0)), zero_expected=float(zero.mean())))
model_comparison = pd.DataFrame(comparison_rows)
residual_frame = founding[["grid_id", "year", "quarter", "period"]].assign(pearson=pearson, mu=mu_founding)
residual_frame["mu_bin"] = pd.qcut(mu_founding, 10, duplicates="drop")
residual_bins = residual_frame.groupby("mu_bin", observed=True).agg(
    rows=("mu", "size"), predicted_mean=("mu", "mean"), mean_residual=("pearson", "mean"),
    residual_variance=("pearson", "var")).reset_index()
adjacent = (residual_frame.year*4 + residual_frame.quarter).groupby(residual_frame.grid_id).diff().eq(1)
lag = residual_frame.groupby("grid_id").pearson.shift()
note("founding", "temporal_residual_correlation", "DESCRIPTIVE",
     str(residual_frame.loc[adjacent, "pearson"].corr(lag.loc[adjacent])))
vif = pd.Series(np.diag(np.linalg.inv(X_founding.drop(columns="const").corr())),
                index=X_founding.columns.drop("const"), name="VIF")
note("founding", "collinearity", "REVIEW" if vif.max() > 10 else "OK", f"maximum VIF={vif.max():.3g}")
note("founding", "extreme_residuals", "REVIEW", f"maximum |Pearson|={np.abs(pearson).max():.3g}; inspect binned fit")
inspection = founding[["grid_id","municipality_id","period","births"]].assign(
    prediction=mu_founding, pearson=pearson)
inspection = inspection.loc[inspection.pearson.abs().nlargest(30).index]
inspection.to_csv(PREPARED_DIR / "largest_residuals_local_only.csv", index=False)
display(model_comparison)
display(residual_bins)

# One targeted functional-form check, prompted by the real run's stock-curvature flag.
stock_frame = founding[[*FOUNDING_TERMS,"period"]].copy()
stock_frame["stock_squared"] = (stock_frame.log_own_firms - stock_frame.log_own_firms.mean())**2
X_stock, X_stock_scaled, stock_scale, _ = design_matrix(stock_frame, [*FOUNDING_TERMS, "stock_squared"], periods=True)
start = pd.Series(founding_fit["params"], index=founding_fit["names"]).reindex([*X_stock.columns, "alpha"], fill_value=0)
start = start.to_numpy() * np.r_[stock_scale, 1.]
stock_fit = count_estimate("nb2", X_stock_scaled, y_founding, founding.grid_id, stock_scale,
                          poisson=poisson_fit, start=start)
note("founding", "stock_sensitivity_fit", "REVIEW" if stock_fit["warnings"] else "OK",
     f"iterations={stock_fit['iterations']}; Newton correction/SE={stock_fit['newton_step_se']:.3g}; warnings={stock_fit['warnings']}")
stock_sensitivity = hypothesis_table(stock_fit, COUNT_TESTS)
stock_sensitivity["model"] = "NB2 with quadratic own-cell stock"
stock_coefficient = coefficient_table(stock_fit).loc["stock_squared"]
note("founding", "stock_curvature", "REVIEW" if stock_coefficient.p_value < .05 else "DESCRIPTIVE",
     f"quadratic stock beta={stock_coefficient.coefficient:.4g}; conditional cell-cluster p={stock_coefficient.p_value:.4g}")
display(stock_sensitivity)
del stock_frame, X_stock, X_stock_scaled, stock_fit, start

In [ ]:
from scipy.spatial import cKDTree
from scipy.sparse import csr_matrix
from statsmodels.stats.multitest import multipletests

moran_rows = []
rng = np.random.default_rng(42)
for period, group in residual_frame.groupby("period", sort=True):
    if period not in ["2016Q4", "2020Q2", "2020Q4", "2023Q3", "2025Q4"]:
        continue
    data = group[["grid_id", "pearson"]].merge(grid, on="grid_id", validate="one_to_one")
    xy = data[["centroid_x", "centroid_y"]].to_numpy()
    pairs = cKDTree(xy).query_pairs(CELL_SIZE*np.sqrt(2)+.01, output_type="ndarray")
    n = len(data)
    if n < 4 or len(pairs) == 0:
        moran_rows.append(dict(period=period, n=n, islands=n, I=np.nan, p_perm=np.nan))
        continue
    W = csr_matrix((np.ones(2*len(pairs)), (np.r_[pairs[:,0], pairs[:,1]],
                    np.r_[pairs[:,1], pairs[:,0]])), shape=(n,n))
    degree = np.asarray(W.sum(axis=1)).ravel()
    W = W.multiply((1/np.maximum(degree,1))[:,None]).tocsr()
    z = data.pearson.to_numpy() - data.pearson.mean()
    denominator = z @ z
    if denominator <= 0:
        moran_rows.append(dict(period=period, n=n, islands=int((degree==0).sum()), I=np.nan, p_perm=np.nan))
        continue
    factor = n / W.sum()
    observed = float(factor*(z @ (W @ z))/denominator)
    exceed = 0
    for _ in range(999):
        shuffled = rng.permutation(z)
        simulated = float(factor*(shuffled @ (W @ shuffled))/denominator)
        exceed += abs(simulated+1/(n-1)) >= abs(observed+1/(n-1))
    moran_rows.append(dict(period=period, n=n, islands=int((degree==0).sum()),
                           I=observed, p_perm=(exceed+1)/1000))
moran_table = pd.DataFrame(moran_rows)
valid = moran_table.p_perm.notna()
moran_table.loc[valid, "p_holm"] = multipletests(moran_table.loc[valid, "p_perm"], method="holm")[1]
note("founding", "spatial_residual_dependence", "REVIEW" if moran_table.p_holm.lt(.05).any() else "SCREEN ONLY",
     "Queen neighbours; five follow-up quarters including previous flags; 999 permutations; Holm. Screen only.")
display(moran_table[["period", "I", "p_holm"]])

## 4. H3: local demand dependence

Preserve the supplied A/B/C classification. B is outside the prespecified A–C contrast.
Unknown classifications exclude the entire paired observation. The cross-model
covariance is calculated in identical clusters. Both groups use Poisson QMLE with
clustered uncertainty: the rare group outcomes do not require fitting an additional
dispersion parameter. This choice follows the numerical/dispersion diagnostics,
independently of the observed direction of the H3 contrast.

In [ ]:
H3_CLASSIFICATION = {
    "Gewerbe und Handwerk": {
        "A": ["118", "120", "124", "125A", "125B", "127", "128"],
        "B": ["101", *map(str, range(103, 109)), *map(str, range(110, 113)), *map(str, range(114, 118)), "119", "121", "122", "123", "126"],
        "C": ["113", "129"]},
    "Industrie": {"A": [], "B": ["209", "213"],
        "C": [*map(str, range(201, 208)), "210", "211", "212", "215", "216", "217", "218"]},
    "Handel": {"A": ["301", "302", "303", "308", "310", "312", "317"],
        "B": ["304A", "306", "309", "313", "315", "316", "320"],
        "C": ["304B", "305", "307", "311", "314", "318"]},
    "Bank und Versicherung": {"A": [], "B": list(map(str, range(401, 407))), "C": []},
    "Transport und Verkehr": {"A": ["505", "507", "508"], "B": ["502", "503", "506"], "C": ["501", "504"]},
    "Tourismus und Freizeitwirtschaft": {"A": ["601", "603", "605", "606"], "B": ["602", "604"], "C": []},
    "Information und Consulting": {"A": [], "B": ["701", "702", "703", "704", "707", "709"], "C": ["705", "706", "708", "710"]},
}

H3_CLASSIFICATION_ROWS = [{"Fachgruppe_ID": code, "sparte": sector, "demand_group": group}
    for sector, groups in H3_CLASSIFICATION.items() for group, codes in groups.items() for code in codes]

H3_GROUP_MAP = {r["Fachgruppe_ID"]: r["demand_group"] for r in H3_CLASSIFICATION_ROWS}

def normalize_fachgruppe(values):
    # Numeric parquet encodings such as 118.0 normalize to 118; suffixes remain meaningful.
    return values.astype("string").str.strip().str.upper().str.replace(r"\s+", "", regex=True).str.replace(r"^(\d{3})\.0+$", r"\1", regex=True)

def prepare_h3_counts():
    """Align sparse group births with the full existing model support, including genuine zero counts."""
    if H3_BIRTHS_PATH.is_file():
        require_columns(H3_BIRTHS_PATH, ["grid_id", "period", "Fachgruppe_ID", "births"])
        key = grid_column(H3_BIRTHS_PATH)
        sparse = pd.read_parquet(H3_BIRTHS_PATH, columns=[key, "period", "Fachgruppe_ID", "births"]).rename(columns={key: "grid_id"})
        if sparse.duplicated(["grid_id", "period", "Fachgruppe_ID"]).any():
            raise ValueError("Duplicate cell-quarter-Fachgruppe birth rows.")
        source_path = H3_BIRTHS_PATH
    else:
        # Legacy fallback uses existing assignments, not newly generated routing/datasets.
        require_columns(FIRMS_PATH, ["firm_id", "grid_id", "founding_date", "Fachgruppe_ID"])
        key = grid_column(FIRMS_PATH)
        firms = pd.read_parquet(FIRMS_PATH, columns=["firm_id", key, "founding_date", "Fachgruppe_ID"]).rename(columns={key: "grid_id"})
        if firms.firm_id.isna().any() or firms.firm_id.duplicated().any(): raise ValueError("Invalid firm IDs for H3 birth aggregation.")
        dates = pd.to_datetime(firms.founding_date, errors="raise")
        firms["period"] = dates.dt.to_period("Q").astype("string")
        sparse = firms.loc[firms.grid_id.notna() & dates.notna()].groupby(["grid_id", "period", "Fachgruppe_ID"], dropna=False).size().rename("births").reset_index()
        source_path = FIRMS_PATH
    if sparse[["grid_id", "period"]].isna().any().any(): raise ValueError("Missing H3 cell/time identifiers.")
    y = sparse.births.to_numpy(dtype=float)
    if not np.isfinite(y).all() or (y<0).any() or (y != np.floor(y)).any(): raise ValueError("Invalid Fachgruppe founding counts.")
    periods = pd.PeriodIndex(sparse.period, freq="Q")
    if not sparse.period.astype(str).eq(periods.astype(str)).all(): raise ValueError("Noncanonical H3 quarter identifiers.")
    sparse = sparse.loc[(periods.year >= START_YEAR) & (periods.year <= END_YEAR)].copy()
    sparse["Fachgruppe_ID"] = normalize_fachgruppe(sparse.Fachgruppe_ID)
    sparse["demand_group"] = sparse.Fachgruppe_ID.map(H3_GROUP_MAP).fillna("UNCLASSIFIED")
    keys = pd.MultiIndex.from_frame(founding[["grid_id", "period"]])
    aligned = sparse.groupby(["grid_id", "period", "demand_group"]).births.sum().unstack("demand_group", fill_value=0)
    aligned = aligned.reindex(index=keys, columns=["A", "B", "C", "UNCLASSIFIED"], fill_value=0).fillna(0)
    total = founding.births.to_numpy(dtype=float)
    unrecorded = total-aligned.sum(axis=1).to_numpy()
    if (unrecorded < -1e-8).any(): raise ValueError("Fachgruppe births exceed total founding counts on the model support.")
    aligned["UNCLASSIFIED"] += np.maximum(unrecorded, 0)
    # Unknown classifications could belong to A/C. Exclude the entire paired observation instead of inventing a group.
    keep = aligned.UNCLASSIFIED.eq(0).to_numpy()
    audit = sparse.merge(founding[["grid_id", "period"]], on=["grid_id", "period"], how="inner", validate="many_to_one")
    coverage = audit.groupby(["Fachgruppe_ID", "demand_group"], dropna=False).births.sum().reset_index()
    missing_codes = int(np.sum(np.maximum(unrecorded, 0)))
    if missing_codes:
        coverage = pd.concat([coverage, pd.DataFrame([{"Fachgruppe_ID": "<missing from source>", "demand_group": "UNCLASSIFIED", "births": missing_codes}])], ignore_index=True)
    report = {"source": str(source_path.relative_to(PROJECT_DIR)), "source_sha256": file_hash(source_path),
        "classification": H3_CLASSIFICATION_ROWS, "frequency": FOUNDING_FREQUENCY,
        "rows_before": len(founding), "rows_retained": int(keep.sum()), "rows_excluded_unknown": int((~keep).sum()),
        "births_before": int(total.sum()), "births_excluded_unknown_observations": int(total[~keep].sum()),
        "unclassified_births": int(aligned.UNCLASSIFIED.sum()),
        "retained_periods": sorted(founding.loc[keep, "period"].unique().tolist()),
        "counts": {g: int(aligned.loc[keep, g].sum()) for g in ["A", "B", "C"]}}
    return aligned, keep, coverage, report

def h3_directional_scores(X, y, fit, scale, weights):
    """Per-observation influence of one population contrast; no large n-by-p score matrix."""
    from scipy.special import digamma
    w = np.asarray([weights.get(name, 0.) for name in fit["names"]])
    factor = np.r_[scale.to_numpy(), 1.] if fit["family"] == "nb2" else scale.to_numpy()
    # H_scaled^{-1} (w/scale) = scale * V_naive_original * w.
    direction = factor * (np.asarray(fit["naive_covariance"]) @ w)
    parameters = np.asarray(fit["params"])*factor
    alpha = float(parameters[-1]) if fit["family"] == "nb2" else 0.
    influence = np.empty(len(y))
    for first in range(0, len(y), 50000):
        last = min(first+50000, len(y))
        x = X.iloc[first:last].to_numpy()
        observed = np.asarray(y[first:last])
        beta = parameters[:-1] if alpha>0 else parameters
        mu = np.exp(x @ beta)
        influence[first:last] = (x @ direction[:X.shape[1]])*(observed-mu)/(1+alpha*mu)
        if alpha>0:
            size = 1/alpha
            alpha_score = -(digamma(observed+size)-digamma(size)+np.log(size)-np.log(size+mu)-(observed-mu)/(size+mu))/alpha**2
            influence[first:last] += direction[-1]*alpha_score
    if not np.isfinite(influence).all(): raise RuntimeError("Nonfinite H3 influence contributions.")
    return influence

In [ ]:
def cluster_influence(X, y, fit, scale, weights, groups):
    influence = h3_directional_scores(X, y, fit, scale, weights)
    codes, labels = pd.factorize(np.asarray(groups), sort=False)
    if (codes < 0).any() or len(labels) < 2:
        raise ValueError("At least two complete clusters are required.")
    n, p, g = len(y), len(fit["params"]), len(labels)
    adjustment = g / (g - 1) * (n - 1) / (n - p)
    return np.bincount(codes, weights=influence, minlength=g) * np.sqrt(adjustment)


# Larger blocks change uncertainty, not coefficients. Their boundaries remain an assumption.
block_rows = []
for identifier, (weights, expected, coverage) in COUNT_TESTS.items():
    if set(weights) - set(founding_fit["names"]):
        continue
    estimate, se = linear_effect(founding_fit, weights)
    cell_scores = cluster_influence(X_nb_scaled, y_founding, founding_fit, nb_scale, weights, founding.grid_id)
    np.testing.assert_allclose(cell_scores @ cell_scores, se**2, rtol=2e-4, atol=1e-9)
    block_scores = cluster_influence(X_nb_scaled, y_founding, founding_fit, nb_scale, weights, founding.spatial_block)
    block_rows.append(contrast_row(identifier, estimate, block_scores @ block_scores,
                                   expected, coverage, f"{CONFIG['spatial_block_m']}m blocks"))
founding_block_hypotheses = pd.DataFrame(block_rows)

# H3 uses matched Poisson QMLE models: sparse groups do not need an extra alpha parameter.
aligned, keep, h3_coverage, h3_report = prepare_h3_counts()
if keep.all():
    h3_frame = founding
    X_h3, X_h3_scaled, h3_scale = X_founding, X_nb_scaled, nb_scale
else:
    h3_frame = founding.loc[keep].reset_index(drop=True)
    X_h3, X_h3_scaled, h3_scale, _ = design_matrix(h3_frame, FOUNDING_TERMS, periods=True)
h3_fits, h3_rows, h3_group_rows, h3_scores = {}, [], [], {}
for group in ["A", "C"]:
    counts = aligned.loc[keep, group].to_numpy(dtype=float)
    if counts.sum() == 0:
        raise ValueError(f"H3 group {group} has no births.")
    fit = count_estimate("poisson", X_h3_scaled, counts, h3_frame.grid_id, h3_scale)
    h3_fits[group] = fit
    estimate, se = linear_effect(fit, POPULATION)
    h3_group_rows.append(dict(group=group, population_effect=estimate, std_error=se,
                             ci_lower=estimate-1.96*se, ci_upper=estimate+1.96*se,
                             births=int(counts.sum()), family="poisson", alpha=0.,
                             iterations=fit["iterations"], newton_step_se=fit["newton_step_se"],
                             fit_warnings=json.dumps(fit["warnings"])))
    for label, clusters in [("cell", h3_frame.grid_id), ("block", h3_frame.spatial_block)]:
        scores = cluster_influence(X_h3_scaled, counts, fit, h3_scale, POPULATION, clusters)
        h3_scores[group, label] = scores
        if label == "cell":
            np.testing.assert_allclose(scores @ scores, se**2, rtol=2e-4, atol=1e-9)
for label in ["cell", "block"]:
    difference = h3_scores["A", label] - h3_scores["C", label]
    estimate = h3_group_rows[0]["population_effect"] - h3_group_rows[1]["population_effect"]
    h3_rows.append(contrast_row("H3", estimate, difference @ difference, "positive", "tested", label))
h3_population_effects = pd.DataFrame(h3_group_rows)
h3_hypotheses = pd.DataFrame(h3_rows)
display(h3_population_effects)
display(h3_hypotheses)
# Release the multi-million-row count matrices before fitting the survival models.
del X_founding, X_nb_scaled, X_h3, X_h3_scaled, h3_frame, founding, residual_frame
del y_founding, mu_poisson, mu_founding, pearson, aligned, counts

## 5. Überlebensmodell

Age since founding, delayed entry, quarterly intervals (start, stop], Efron ties.
Keep the numerically validated cluster-score calculation: the real software version's
library covariance did not pass its check. This is required statistical code.

In [ ]:
def cox_score_blocks(model, params, groups):
    # Efron case-weight influence: derivative of the weighted partial-likelihood score.
    codes, labels = pd.factorize(np.asarray(groups), sort=False)
    if (codes<0).any(): raise ValueError("Missing Cox cluster identifiers.")
    blocks = np.zeros((len(labels), model.exog.shape[1]))
    martingale = np.zeros(len(model.endog))
    used = np.zeros(len(model.endog), dtype=bool)
    event_rows, schoenfeld = [], []
    surv = model.surv
    for st in range(surv.nstrat):
        x = surv.exog_s[st]
        original = surv.stratum_rows[st]
        eta = x @ params
        if surv.offset_s is not None: eta = eta + surv.offset_s[st]
        w = np.exp(eta-eta.max())
        for k in range(len(surv.ufailt[st]))[::-1]:
            event_time = surv.ufailt[st][k]
            # Explicit membership also handles intervals spanning no event time.
            risk = np.flatnonzero((surv.entry_s[st]<=event_time) & (surv.time_s[st]>=event_time))
            death = surv.ufailt_ix[st][k]
            m = len(death)
            S0 = w[risk].sum()
            S1 = w[risk] @ x[risk]
            E0 = w[death].sum()
            E1 = w[death] @ x[death]
            if model.ties == "efron":
                fraction = np.arange(m)/m
                denominator = S0-fraction*E0
                mean = (S1[None,:]-fraction[:,None]*E1)/denominator[:,None]
                h = np.sum(1/denominator)
                hx = np.sum(mean/denominator[:,None], axis=0)
                hf = np.sum(fraction/denominator)
                hxf = np.sum(fraction[:,None]*mean/denominator[:,None], axis=0)
                average = mean.mean(axis=0)
            else:
                average = S1/S0
                h, hx, hf, hxf = m/S0, m/S0*average, 0., np.zeros(x.shape[1])
            score_risk = -w[risk,None]*(x[risk]*h-hx)
            score_death = x[death]-average + w[death,None]*(x[death]*hf-hxf)
            np.add.at(blocks, codes[original[risk]], score_risk)
            np.add.at(blocks, codes[original[death]], score_death)
            martingale[original[risk]] -= w[risk]*h
            martingale[original[death]] += 1+w[death]*hf
            used[original[risk]] = True
            event_rows.extend(original[death])
            schoenfeld.extend(x[death]-average)
    return blocks, np.asarray(event_rows, dtype=int), np.asarray(schoenfeld), martingale, used

def verify_cox_cluster_implementation():
    # Independent small weighted-score calculation: includes ties, entry and strata.
    x = np.array([[0.,1.],[1.,0.],[.5,.4],[-.3,1.2],[1.1,.2],[.2,-.4],
                  [.1,.8],[.7,.1],[-.4,.3],[1.3,.7],[.4,-.2],[.9,.5],[.1,.3],[.6,.5]])
    stop = np.array([1,2,2,3,4,4,1,2,3,3,4,5,2.,3.], dtype=float)
    entry = np.nextafter(np.array([0,0,1,0,2,3,0,0,1,2,0,4,0.,2.1], dtype=float), np.inf)
    event = np.array([0,1,1,0,1,0,1,0,1,1,0,1,0,0])
    strata = np.r_[np.zeros(6), np.ones(6), 2, 0]
    groups = np.array([0,1,1,2,3,3,4,4,5,5,6,6,7,8])
    params = np.array([.15, -.25])
    library_valid = True
    for ties in ["efron", "breslow"]:
        model = PHReg(stop, x, status=event, entry=entry, strata=strata, ties=ties)
        blocks, _, _, _, _ = cox_score_blocks(model, params, groups)
        def weighted_score(case_weights):
            total = np.zeros(2)
            for stratum in np.unique(strata):
                for t in np.unique(stop[(strata==stratum) & (event==1)]):
                    risk = (strata==stratum) & (entry<=t) & (stop>=t)
                    died = (strata==stratum) & (stop==t) & (event==1)
                    hazard = case_weights*np.exp(x@params)
                    S0, S1 = hazard[risk].sum(), hazard[risk]@x[risk]
                    E0, E1 = hazard[died].sum(), hazard[died]@x[died]
                    m = int(died.sum())
                    f = np.arange(m)/m if ties=="efron" else np.zeros(m)
                    total += case_weights[died]@x[died]
                    total -= case_weights[died].sum()/m*np.sum((S1[None,:]-f[:,None]*E1)/(S0-f*E0)[:,None], axis=0)
            return total
        numerical = []
        for group in pd.unique(groups):
            delta = groups==group
            numerical.append((weighted_score(1+1e-5*delta)-weighted_score(1-1e-5*delta))/(2e-5))
        np.testing.assert_allclose(blocks, numerical, rtol=2e-5, atol=1e-8)
        np.testing.assert_allclose(blocks.sum(axis=0), model.score(params), rtol=1e-8, atol=1e-8)
        bread = np.linalg.inv(-model.hessian(params))
        checked = bread @ (blocks.T@blocks) @ bread
        model.groups = groups
        library = model.robust_covariance(params)
        library_valid &= bool(np.isfinite(library).all() and np.allclose(library, checked, rtol=1e-6, atol=1e-8))
    print("Cox CR0: finite-difference cluster-score checks passed; library covariance validated:", library_valid)
    return library_valid

_cox_diagnostic_state = None
def make_cox_model(ties="efron"):
    return PHReg(survival_frame.stop.to_numpy(dtype=float), X_cox_scaled,
        status=survival_frame.event.to_numpy(), entry=np.nextafter(survival_frame.start.to_numpy(dtype=float), np.inf),
        strata=survival_frame.Fachgruppe_ID.to_numpy(), ties=ties)

def estimate_cox(model, library_valid, retain=True):
    global _cox_diagnostic_state
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter("always")
        # Same PHReg likelihood; retain raw optimizer status that PHReg.fit's wrapper discards.
        optimizer_steps = []
        raw = LikelihoodModel.fit(model, method="bfgs", maxiter=CONFIG["maxiter_cox"], gtol=1e-9, disp=0,
                                  callback=lambda _: optimizer_steps.append(None))
        if not raw.mle_retvals.get("converged", False): raise RuntimeError("Cox optimizer did not converge within maxiter_cox.")
        params = np.asarray(raw.params)
        _, step_se = numerical_check(model, params)
        blocks, event_rows, sch, martingale, used = cox_score_blocks(model, params, survival_frame.grid_id)
        score_error = float(np.max(np.abs(blocks.sum(axis=0)-model.score(params))))
        if not np.allclose(blocks.sum(axis=0), model.score(params), rtol=1e-7, atol=1e-6):
            raise RuntimeError("Cox cluster-score identity failed; robust inference is unavailable.")
        bread = np.linalg.inv(-model.hessian(params))
        covariance = bread @ (blocks.T@blocks) @ bread
        covariance_method = "validated Efron/Breslow case-weight cluster-score sandwich CR0"
        if library_valid:
            model.groups = survival_frame.grid_id.to_numpy()
            library_covariance = model.robust_covariance(params)
            if np.isfinite(library_covariance).all() and np.allclose(library_covariance, covariance, rtol=1e-5, atol=1e-7):
                covariance = library_covariance
                covariance_method = "validated statsmodels PHReg group covariance"
        check_fit_arrays(params, covariance, model.loglike(params))
    messages = sorted({str(w.message) for w in caught})
    for message in messages: print("Cox warning:", message)
    factor = cox_scale.to_numpy()
    cell_labels = pd.unique(survival_frame.grid_id)
    block_codes, block_labels = pd.factorize(pd.Series(cell_labels).map(block_map), sort=False)
    if (block_codes < 0).any() or len(block_labels) < 2:
        raise ValueError("Insufficient spatial blocks for Cox sensitivity.")
    spatial_scores = np.zeros((len(block_labels), blocks.shape[1]))
    np.add.at(spatial_scores, block_codes, blocks)
    spatial_covariance = bread @ (spatial_scores.T @ spatial_scores) @ bread
    if retain: _cox_diagnostic_state = (blocks, event_rows, sch, martingale, used)
    return {"family": "cox", "names": list(X_cox.columns), "params": (params/factor).tolist(),
        "covariance": (covariance/np.outer(factor, factor)).tolist(), "naive_covariance": (bread/np.outer(factor, factor)).tolist(),
        "block_covariance": (spatial_covariance/np.outer(factor, factor)).tolist(),
        "newton_step_se": step_se, "n_spatial_blocks": len(block_labels),
        "log_likelihood": float(model.loglike(params)), "aic": float(-2*model.loglike(params)+2*len(params)),
        "converged": True, "optimizer": "BFGS", "iterations": len(optimizer_steps),
        "function_evaluations": int(raw.mle_retvals.get("fcalls", 0)), "warnings": messages,
        "n_observations": int(len(survival_frame)), "n_clusters": int(survival_frame.grid_id.nunique()),
        "n_events": int(survival_frame.event.sum()), "n_informative_intervals": int(used.sum()), "covariance_method": covariance_method,
        "score_identity_max_error": score_error, "rows_outside_event_risk_sets": int((~used).sum()),
        "ties": model.ties, "entry_convention": "(start, stop]; nextafter(start,+inf)"}

In [ ]:
X_cox, X_cox_scaled, cox_scale, cox_constants = design_matrix(survival_frame, SURVIVAL_TERMS)
library_covariance_valid = verify_cox_cluster_implementation()
cox_model = make_cox_model()
cox_fit = estimate_cox(cox_model, library_covariance_valid)
cox_coefficients = coefficient_table(cox_fit, "hazard_ratio")
cox_hypotheses = hypothesis_table(cox_fit, SURVIVAL_TESTS)
block_fit = dict(cox_fit, covariance=cox_fit["block_covariance"])
cox_block_hypotheses = hypothesis_table(block_fit, SURVIVAL_TESTS).assign(covariance="spatial blocks")
display(cox_hypotheses)
display(cox_block_hypotheses)
display(cox_coefficients[["coefficient", "std_error", "hazard_ratio", "hr_ci_lower", "hr_ci_upper"]])

In [ ]:
blocks, event_rows, sch, martingale, used = _cox_diagnostic_state
ph_rows, ph_bins = [], []
event_age = survival_frame.stop.to_numpy()[event_rows]/4
for j, term in enumerate(X_cox.columns):
    correlation = stats.spearmanr(np.log1p(event_age), sch[:,j])
    ph_rows.append(dict(term=term, spearman_rho=correlation.statistic, screen_p_value=correlation.pvalue))
    binned = pd.DataFrame({"age_years": event_age, "residual": sch[:,j]})
    binned["age_band"] = pd.cut(event_age, [0,2,5,10,20,np.inf])
    summary = binned.groupby("age_band", observed=True).agg(n_events=("residual","size"),
        mean_age_years=("age_years","mean"), mean_schoenfeld=("residual","mean")).reset_index()
    summary["age_band"] = summary.age_band.astype(str)
    summary["term"] = term
    ph_bins.append(summary.loc[summary.n_events.ge(5)])
ph_screen = pd.DataFrame(ph_rows)
valid = ph_screen.screen_p_value.notna()
ph_screen.loc[valid, "p_holm"] = multipletests(ph_screen.loc[valid,"screen_p_value"], method="holm")[1]
note("survival", "proportional_hazards", "REVIEW" if ph_screen.p_holm.lt(.05).any() else "SCREEN ONLY",
     "Schoenfeld/event-age correlations are a screen, not a cluster-robust formal PH test.")
display(ph_screen)

# One prespecified time-varying-coefficient sensitivity, rather than searching age cutoffs.
age_effects = []
if CONFIG["ph_sensitivity"]:
    base_X, base_scaled, base_scale = X_cox, X_cox_scaled, cox_scale
    base_state = _cox_diagnostic_state
    log_age = np.log1p(survival_frame.stop.to_numpy()/4)
    varying = pd.concat([base_X, base_X.mul(log_age, axis=0).add_suffix("_log_age")], axis=1)
    X_cox, X_cox_scaled, cox_scale, _ = design_matrix(varying, list(varying.columns))
    age_fit = estimate_cox(make_cox_model(), library_covariance_valid, retain=False)
    note("survival", "age_sensitivity_fit", "REVIEW" if age_fit["warnings"] else "OK",
         f"iterations={age_fit['iterations']}; Newton correction/SE={age_fit['newton_step_se']:.3g}; warnings={age_fit['warnings']}")
    for identifier, (weights, expected, coverage) in SURVIVAL_TESTS.items():
        for years in [1,5,10]:
            contrast = dict(weights)
            contrast.update({name+"_log_age": value*np.log1p(years) for name,value in weights.items()})
            if set(contrast)-set(age_fit["names"]):
                continue
            for covariance in ["cell", "block"]:
                use_fit = age_fit if covariance=="cell" else dict(age_fit, covariance=age_fit["block_covariance"])
                estimate, se = linear_effect(use_fit, contrast)
                row = contrast_row(identifier, estimate, se**2, expected, coverage, covariance)
                row["age_years"] = years
                age_effects.append(row)
    X_cox, X_cox_scaled, cox_scale = base_X, base_scaled, base_scale
    _cox_diagnostic_state = base_state
    del varying, log_age
age_effects = pd.DataFrame(age_effects)
display(age_effects)

## 6. Export

H4 covers only lines/no-stop proxies; H5 covers only the common motorway-time effect.
Do not call the complete hypotheses confirmed. A spatial-cluster sensitivity is
not a general cure for all spatial dependence. Age-varying effects remain conditional
on that specification. All estimates describe associations.

In [ ]:
locations = survival_frame.groupby("standort_id").agg(
    entry=("start","min"), exit=("stop","max"), event=("event","max"))

manifest.update(status="completed", sample_description=sample_rows, diagnostics=diagnostic_rows,
    founding={k:v for k,v in founding_fit.items() if k not in ["params","covariance","naive_covariance","names"]},
    survival={k:v for k,v in cox_fit.items() if k not in ["params","covariance","block_covariance","naive_covariance","names"]},
    h3={k:v for k,v in h3_report.items() if k != "classification"})
manifest["sample_summary"] = {
    "locations": len(locations), "exits": int(locations.event.sum()),
    "right_censored": int(locations.event.eq(0).sum()),
    "left_truncated": int(locations.entry.gt(0).sum()),
}
tables = {
    "sample_description": pd.DataFrame(sample_rows),
    "founding_nb2_results": founding_coefficients,
    "founding_nb2_hypotheses": founding_hypotheses,
    "founding_block_hypotheses": founding_block_hypotheses,
    "founding_stock_sensitivity": stock_sensitivity,
    "model_comparison": model_comparison,
    "founding_residual_bins": residual_bins,
    "founding_moran_screen": moran_table,
    "h3_population_effects": h3_population_effects,
    "h3_group_diagnostics": h3_population_effects,
    "h3_hypotheses": h3_hypotheses,
    "h3_classification_coverage": h3_coverage,
    "h3_fachgruppe_classification": pd.DataFrame(H3_CLASSIFICATION_ROWS),
    "survival_cox_results": cox_coefficients,
    "survival_cox_hypotheses": cox_hypotheses,
    "survival_block_hypotheses": cox_block_hypotheses,
    "survival_age_effects": age_effects,
    "survival_schoenfeld_screen": ph_screen,
    "survival_schoenfeld_bins": pd.concat(ph_bins, ignore_index=True),
    "diagnostics": pd.DataFrame(diagnostic_rows),
}
for name, table in tables.items():
    table.to_csv(BUNDLE_DIR / f"{name}.csv", index=name in ["founding_nb2_results","survival_cox_results"])
# Counts, estimates and diagnostics are aggregate exports; no model rows or influence arrays.
manifest["outputs"] = sorted(p.name for p in BUNDLE_DIR.iterdir() if p.suffix==".csv")
(BUNDLE_DIR / "model_run_manifest.json").write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False, default=lambda x: x.item() if hasattr(x,"item") else str(x)),
    encoding="utf-8")
pointer = json.dumps({"run_id": RUN_ID, "status": "completed"})
for name in ["latest_attempt.json","latest_run.json"]:
    (RESULT_DIR / name).write_text(pointer, encoding="utf-8")
print("Completed:", BUNDLE_DIR)
print("Use notebook 10 for results. Treat PH, spatial and stock-form sensitivity as part of interpretation.")